In [ ]:
!pip install -q openai-agents==0.2.2 python-dotenv requests

import os
import requests                                # sends HTTP requests to the Tavily API
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv()
import os, requests
from openai import OpenAI
from IPython.display import display, Markdown
from google.colab import userdata

openai_api_key = userdata.get("OPENAI_API_KEY")
tavily_api_key = userdata.get("TAVILY_API_KEY")
os.environ["OPENAI_API_KEY"] = openai_api_key     # the Agents SDK reads the key from here
openai_client = OpenAI(api_key = openai_api_key)
print(openai_api_key[:5], tavily_api_key[:5])



sk-pr tvly-


In [ ]:
import json                                   # handle JSON data
from typing_extensions import TypedDict       # describe the shape of a dictionary
from agents import function_tool              # turns a function into an agent tool


In [ ]:
class TavilySearchParams(TypedDict):
    query: str         # the search text
    max_results: int   # how many results to return

In [ ]:
@function_tool
def tavily_search(params: TavilySearchParams) -> str:
    """
    Calls the Tavily API and returns a string summary of top search results.
    """
    url = "https://api.tavily.com/search"              # Tavily endpoint
    headers = {"Content-Type": "application/json"}     # we are sending JSON
    payload = {
        "api_key": tavily_api_key,                      # who we are
        "query": params["query"],                       # what to search
        "max_results": params.get("max_results", 2),    # how many results
    }
    response = requests.post(url, json = payload, headers = headers)

    if response.status_code == 200:                     # 200 = success
        results = response.json().get("results", [])
        summary = "\n".join([f"{i+1}. {r['title']}: {r['content']}" for i, r in enumerate(results)])
        return summary if summary else "No relevant results found."
    else:
        return f"Tavily API error: {response.status_code}"



In [ ]:
def print_markdown(text):
    """Displays text as Markdown in Jupyter."""
    display(Markdown(text))

In [ ]:
from agents import SQLiteSession, Agent, Runner
session = SQLiteSession("live_researcher_practice")       # memory from Project 2

live_researcher_agent = Agent(name = "Live Market Researcher",
                              instructions = """
CONTEXT:
You are a world-class market research assistant with access to real-time web search via the tavily_search tool.

INSTRUCTION:
- Analyze the user's question and determine if recent or real-time information is needed.
- If the question involves recent events, news, or product info, always call tavily_search.
- Summarize search results clearly and concisely, do not copy-paste.
- Always start your answer with: "🔍 According to a web search …"

INPUT:
You will receive a conversation history and the latest user question. Use the full context to inform your response.

OUTPUT:
Provide a clear, well-structured answer that references the search results when appropriate.
""",
                              model = "gpt-4.1-mini",
                              tools = [tavily_search])          # <- NEW

q1 = "What are people saying about the new GPT-5 Model?"
run1 = await Runner.run(starting_agent = live_researcher_agent, input = q1, session = session)
print_markdown(f"### 🤖 Agent's Answer\n{run1.final_output}")



### 🤖 Agent's Answer
🔍 According to a web search, people are saying very positive things about the new GPT-5 model. It is considered OpenAI's best model yet, excelling in areas like health-related questions, coding, design, and complex knowledge work across many professions. Users and reviewers highlight that GPT-5 acts more like an active thought partner, proactively flagging issues and asking clarifying questions.

The model shows significant improvements in generating high-quality, responsive websites and apps with better design sensibility. It's also praised for reduced hallucination rates, meaning it fabricates fewer answers and offers safer, more reliable completions. GPT-5 handles tasks more accurately by recognizing when it cannot complete a task and clearly explaining its limitations.

Reviewers mention that GPT-5 is faster, smarter, and more useful across writing, coding, healthcare, and economic knowledge work. It is accessible to all ChatGPT users, including the free tier, and is priced competitively in the API space. Early testers appreciate its improved accuracy, safety measures, and the ability to be a dependable daily driver for a variety of professional and creative tasks.

In summary, GPT-5 is seen as a significant upgrade over previous models, combining advanced capabilities with safer, clearer, and more reliable AI interactions.

In [ ]:
session = SQLiteSession("live_researcher_practice")

q1 = "What do reviewers say about the new Tesla Cybertruck?"
run1 = await Runner.run(live_researcher_agent, q1, session = session)
print_markdown(f"**Answer:** {run1.final_output}")

q2 = "Summarize the main features in one paragraph"
run2 = await Runner.run(live_researcher_agent, q2, session = session)
print_markdown(f"**Answer:** {run2.final_output}")

**Answer:** 🔍 According to a web search, reviewers of the new Tesla Cybertruck generally have very positive impressions, highlighting it as a technical and engineering marvel with futuristic appeal. 

- One detailed review from CleanTechnica praised the Cybertruck for its smooth and quiet ride, ease of steering, strong acceleration, and driving dynamics that surpass Tesla's Model S. The reviewer expects it to be a top-selling electric vehicle in its class.
- Expert reviews note the impressive off-road capabilities of the Cybertruck, citing features like adjustable air suspension on premium models, high ground clearance, and strong approach and departure angles.
- The vehicle's interior is described as minimalist and spartan, continuing Tesla’s design philosophy. The rear seat space is generous, though taller passengers might find it a bit tight.
- Long-term ownership reviews mention the experience of the novelty wearing off over time but also note positive aspects such as reliability, driving experience, and overall satisfaction after a year or more.
- Community forums reflect a mix of excitement and constructive feedback, with users sharing both positives and negatives based on extended use.

Overall, the Cybertruck is seen as a bold and futuristic electric truck delivering strong performance and unique styling, with some expected trade-offs in interior comfort and refinement.

**Answer:** The Tesla Cybertruck features a futuristic, angular design with a highly durable exoskeleton. It offers strong performance with quick acceleration, smooth and quiet ride quality, and superior steering ease compared to other Tesla models. The premium versions come equipped with adjustable air suspension that provides up to 16 inches of ground clearance, making it capable off-road with impressive approach and departure angles. The interior is minimalist and spacious for a truck, with heated and ventilated seats and dual touchscreens in higher trims. It is built for both everyday driving and rugged use, combining electric efficiency with utility and advanced technology.

In [ ]:
from agents import CodeInterpreterTool

code_interpreter = CodeInterpreterTool(
    tool_config = {"type": "code_interpreter",
                   "container": {"type": "auto"}})     # OpenAI creates a sandbox automatically

analyst_agent = Agent(name = "Analyst Agent",
                      instructions = """...use tavily_search for real-time facts,
                                        then use code_interpreter for calculations, or both...""",
                      model = "gpt-4.1-mini",
                      tools = [tavily_search, code_interpreter])   # one custom + one built-in tool

session = SQLiteSession("live_researcher")
q1 = ("Find the current price of a Tesla Cyber Truck in Canada. "
      "Then with the help of code_interpreter, simulate how the price would change if import tariffs increased from 5% to 20% in 1% increments. "
      "For each tariff rate, calculate the new price using Python and provide a summary of the results across all scenarios.")
run1 = await Runner.run(starting_agent = analyst_agent, input = q1, session = session)



In [ ]:
print_markdown(f"**Answer:** {run1.final_output}")

**Answer:** The current price of the Tesla Cybertruck in Canada is approximately CAD 139,990 for the Premium All-Wheel Drive trim.

Now, I will simulate how the price would change if import tariffs increased from 5% to 20% in 1% increments, calculating the new price for each tariff level assuming the base price is CAD 139,990 and the tariff is applied on top of the base price.

In [ ]:
run2 = await Runner.run(analyst_agent,
                        "Go ahead: run the tariff calculation now using code_interpreter and show the table.",
                        session = session)
print_markdown(f"**Answer:** {run2.final_output}")

**Answer:** Here is the table showing how the price of a Tesla Cybertruck in Canada would change as import tariffs increase from 5% to 20% in 1% increments:

| Tariff Rate (%) | Price with Tariff (CAD) |
|-----------------|------------------------|
| 5               | 107,100.00             |
| 6               | 108,120.00             |
| 7               | 109,140.00             |
| 8               | 110,160.00             |
| 9               | 111,180.00             |
| 10              | 112,200.00             |
| 11              | 113,220.00             |
| 12              | 114,240.00             |
| 13              | 115,260.00             |
| 14              | 116,280.00             |
| 15              | 117,300.00             |
| 16              | 118,320.00             |
| 17              | 119,340.00             |
| 18              | 120,360.00             |
| 19              | 121,380.00             |
| 20              | 122,400.00             |

The price increases linearly by CAD 1,020 for each 1% increase in tariff rate. If you want, I can provide further analysis or visualization as well.